# 04 · Analysis and figures

Everything the report needs, built from `results/*.json` alone. **No Drive, no GPU, no model** —
this notebook runs on a laptop in a couple of seconds, which is also what makes the figures
cheap to regenerate when a number changes.

Plotting lives in `src/figures.py`, not in these cells, so the figures can be rebuilt from a
script or a test without opening a notebook. Each figure is printed with the table behind it:
the numbers are the record, the figure is the argument.

Figures are written to `results/figures/` as **PNG** (for the repository) and **PDF** (for the
manuscript, where you want vectors).

## 0 · Setup

In [ ]:
# Runs locally from the repo root. In Colab, clone first and the rest is identical.
import os
if not os.path.exists('src'):
    REPO = '/content/hausa-med-qa'
    if not os.path.exists(REPO):
        !git clone -q https://github.com/avvas200/hausa-med-qa.git {REPO}
    %cd {REPO}

import json
import pandas as pd
from src import figures as FIG

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

FIG.style()
res = FIG.load()
print('baseline  :', res['baseline']['primary']['n_items'], 'items')
print('E1        :', res['e1']['n_rows'], 'coded rows')
print('H4 gate   :', 'present' if res['h4_gate'] else 'not run yet')
print('H4 results:', 'present' if res['h4'] else 'not run yet')

## 1 · The headline

The gap is the obvious finding. The row that matters more is the last two columns: what the
Hausa score is **relative to a model that reads nothing and always answers the most common
letter**, and how often the answer could be parsed at all.

In [ ]:
t1 = FIG.table_headline(res)
display(t1)

base = res['baseline']['majority_letter_baseline']['p0']
below = t1[t1['HA vs baseline'] < 0]['model'].tolist()
print(f'majority-letter baseline (canonical order): {base:.3f}')
print(f'at or below it in Hausa: {len(below)} of 3 -> ' + ', '.join(below))

FIG.fig_accuracy(res, 'results/figures/fig1_accuracy.png');

## 2 · The pre-registered effects

Three quantities on three different scales, so three panels. A shared axis would flatten the
difference between an accuracy gap and a recovered fraction, and a second y-axis would be worse.

H2 fails for MedGemma: its interval crosses zero, so back-translation does not demonstrably
recover its gap. That is the one decision rule of the nine that is not met, and it is reported
as such rather than folded into the others.

In [ ]:
display(FIG.table_decisions(res))
FIG.fig_effects(res, 'results/figures/fig2_effects.png');

## 3 · Format following, and how much the scoring rule carries

Qwen parses only **0.47** of its Hausa answers under the primary (text-parsed) rule, against
1.00 for the other two. Its Hausa collapse is therefore not purely a knowledge failure, and the
next figure prices that: scoring the same outputs by letter probability roughly doubles Qwen's
Hausa accuracy, 0.122 to 0.266, while barely moving Llama or MedGemma.

This is the strongest argument in the study for reporting both scoring rules. A paper that
reported only the parsed rule would overstate Qwen's deficit; one that reported only letter
probability would hide a real format failure.

In [ ]:
pr = pd.DataFrame(res['baseline']['parse_rate']).T[['en_p0', 'ha_p0']]
pr.index = [FIG.LABEL[m] for m in pr.index]
display(pr.rename(columns={'en_p0': 'parse rate EN', 'ha_p0': 'parse rate HA'}))
FIG.fig_parse_rate(res, 'results/figures/fig3_parse_rate.png');

In [ ]:
display(FIG.table_sensitivity(res))
FIG.fig_scoring(res, 'results/figures/fig4_scoring.png');

## 4 · E1: why the items fail

Exploratory, no hypothesis attached. Pooled over both models; per-model proportions are
descriptive only because the re-code was unblinded. Read this as **single-coder coding under a
written codebook with machine-enforced consistency rules, plus a second-reader verification
pass** — see `PREREGISTRATION.md` § 13, deviation 5. No κ is claimed.

The finding that survives inspection: **18 of the 34 text errors sit on the question stem**, the
longest translated span and the one human validation checked most closely. The superseded first
pass placed none there.

In [ ]:
p = res['e1']['primary_pooled']
rows = [{'category': c, 'n': v['n'], 'proportion': v['prop'],
         '95% Wilson': f"[{v['ci95'][0]:.3f}, {v['ci95'][1]:.3f}]"}
        for c, v in p['codes'].items()]
display(pd.DataFrame(rows).sort_values('n', ascending=False))
print('text errors by location :', p['text_error_location'])
print('segment review status   :', p['text_error_segment_status'])
print('rows with an untranslated option:',
      res['e1']['untranslated_options']['rows_with_any_identical_option'], 'of 50')

FIG.fig_e1(res, 'results/figures/fig5_e1_errors.png');

## 5 · H4

Filled in once the pilot gate (notebook 03a) and, if it says go, the intervention (notebook 03)
have run. Until then this section states plainly that H4 is outstanding rather than quietly
omitting it.

In [ ]:
if res['h4_gate'] is None:
    print('H4 pilot gate has not been run. Run notebooks/03a_dpo_pilot.ipynb on a GPU.')
    print('Before booking one: PYTHONPATH=. python tests/preflight_h4.py')
else:
    g = res['h4_gate']
    print(f"parse rate            {g['parse_rate']:.3f}")
    print(f"accuracy, all samples {g['accuracy_all_samples']:.3f}")
    print(f"pairs {g['pairs']}/{g['n_items']} "
          f"-> projected {g['projected_pairs_full_pool']} "
          f"vs threshold {g['min_pairs_threshold']}")
    print(f"estimated luck share of chosen {g['estimated_luck_share_of_chosen']:.3f}")

if res['h4'] is None:
    print('\nH4 intervention not run.')

## 6 · Export

Regenerates all five figures from scratch and writes the paper tables as CSV, so the manuscript
never transcribes a number by hand.

In [ ]:
paths = FIG.all_figures(res)
for name, p in paths.items():
    print('figure ', p)

tables = {'table1_headline': FIG.table_headline(res),
          'table2_decisions': FIG.table_decisions(res),
          'table3_sensitivity': FIG.table_sensitivity(res)}
for name, df in tables.items():
    out = f'results/figures/{name}.csv'
    df.to_csv(out, index=False, encoding='utf-8-sig')
    print('table  ', out)

## What the figures say

1. **All three models lose most of their medical accuracy in Hausa** — 19 to 40 points, every
   gap significant after Holm correction.
2. **Two of the three fall to or below a trivial baseline.** Llama (0.276) and Qwen (0.122) sit
   under the 0.318 majority-letter baseline; only MedGemma (0.356) stays above it, by 3.8 points.
   A reader who takes only one number from this study should take that one.
3. **Back-translating to English recovers about half the gap** for Llama and two thirds for
   Qwen, but not demonstrably for MedGemma. So the deficit is substantially, though not wholly,
   about the language of presentation rather than the medical content.
4. **Answers are far less stable in Hausa** under option reordering, for every model.
5. **Where the Hausa text is the cause, it is usually medical vocabulary** (0.58, 95% CI
   0.44–0.71), and usually in the question stem.

The honest caveat to carry into the report: the Hausa set is machine-translated, and E1 says
most discordance traces to the translation rather than to the model. The paper should frame the
gap as **the gap a Hausa speaker would encounter using these models through the best freely
available translation**, which is the practically relevant quantity, rather than as a clean
measure of multilingual reasoning. The human validation subset (mean adequacy 4.52 and 4.36)
bounds how much of it is translation noise, and both sensitivity analyses leave H1 unchanged.